In [41]:
#FILE DI COPIA PER SICUREZZA

In [42]:
import SPARQLWrapper
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import requests
import time 
from tqdm import tqdm

In [43]:
#carichiamo il csv con i soggetti di Wikidata
df_wikidata = pd.read_csv("wd_dataset.csv", encoding="utf-8")

df_wikidata.head(3)


,entity,label,aliases,description,category
0,http://www.wikidata.org/entity/Q279782,Abarbarea,NaN,naiade della mitologia greca,greek_deity
1,http://www.wikidata.org/entity/Q2370052,Acanto,NaN,"personaggio della mitologia greca, probabilmen...",greek_deity
2,http://www.wikidata.org/entity/Q4668463,Acaste,NaN,Oceanina nella mitologia greca,greek_deity


In [44]:
#estraggo tutte le label di wikidata 
labels = (
    df_wikidata["label"]
    .dropna() #tolgo i valori nulli
    .str.strip() #tolgo gli spazi bianchi all'inizio e alla fine
    .tolist() #converto in lista

)

#estraggo gli aliases di wikidata separando quelli multipli
aliases = []
for alias_str in df_wikidata["aliases"].dropna(): #per ogni stringa di alias che non è nulla
    alias_list = [alias.strip() for alias in alias_str.split(",")] #separo gli alias con la virgola, tolgo gli spazi e metto tutto in minuscolo
    aliases.extend(alias_list) #aggiungo alla lista totale degli alias

#unisco tutto togliendo i duplicati
myth_terms = list(set(labels + aliases)) 
len(myth_terms)

2928

In [45]:
#ENDPOINT SPARLQ DI ARCO 
arco_endpoint = "https://dati.cultura.gov.it/sparql"

sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

In [46]:
# NUMERO TOTALE DEI BENI STORICO ARTISTICI - senza duplicati 
query1 = """
PREFIX arco: <https://w3id.org/arco/ontology/arco/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 

SELECT (COUNT(DISTINCT ?item) AS ?totalItems)
WHERE {
  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
}
"""


sparql_arco.setQuery(query1)
results = sparql_arco.query().convert()

for result in results["results"]["bindings"]:
    print(f"Total Historic or Artistic Properties: {result['totalItems']['value']}")

Total Historic or Artistic Properties: 2259266


In [47]:
#query pr estrarre subject e label dei beni storico artistici in batch (altrimenti esplode)
# Iterare su tutti i 2.258.640 beni con batch di 20000 (altrimenti esplode)

rows = []
batch_size = 20000
total_items = 2258640

for offset in range(0, total_items, batch_size):
    print(f"Processando batch: offset={offset}, items processati={len(rows)}")
    
    query2 = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

SELECT ?item ?subject ?label
WHERE {{

  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
  
  OPTIONAL {{ ?item rdfs:label ?label . }}

  OPTIONAL {{ ?item a-cd:subject ?subject . }}

}}
LIMIT {batch_size}
OFFSET {offset}
"""
    
    sparql_arco.setQuery(query2)
    results = sparql_arco.query().convert()
    
    # estrai i dati da questo batch
    for r in results["results"]["bindings"]:
        rows.append({
            "item": r["item"]["value"] if "item" in r else None,
            "label": r["label"]["value"] if "label" in r else None,
            "subject": r["subject"]["value"] if "subject" in r else None
        })
    
    # se il batch è vuoto, ferma il loop
    if len(results["results"]["bindings"]) < batch_size:
        print(f"Batch incompleto ({len(results['results']['bindings'])} items), fine loop")
        break


Processando batch: offset=0, items processati=0
Processando batch: offset=20000, items processati=20000
Processando batch: offset=40000, items processati=40000
Processando batch: offset=60000, items processati=60000
Processando batch: offset=80000, items processati=80000
Processando batch: offset=100000, items processati=100000
Processando batch: offset=120000, items processati=120000
Processando batch: offset=140000, items processati=140000
Processando batch: offset=160000, items processati=160000
Processando batch: offset=180000, items processati=180000
Processando batch: offset=200000, items processati=200000
Processando batch: offset=220000, items processati=220000
Processando batch: offset=240000, items processati=240000
Processando batch: offset=260000, items processati=260000
Processando batch: offset=280000, items processati=280000
Processando batch: offset=300000, items processati=300000
Processando batch: offset=320000, items processati=320000
Processando batch: offset=340000

In [48]:

# creiamo il dataframe consolidato
df_arco = pd.DataFrame(rows)
df_arco = df_arco.drop_duplicates(subset="item")

print(f"Righe totali in df_arco : {len(df_arco)}")
df_arco[["item", "label", "subject"]].head(1)

Righe totali in df_arco : 1120196


,item,label,subject
0,https://w3id.org/arco/resource/HistoricOrArtis...,Arlecchino primordiale. Fine del cinquecento. ...,"figurino per costume di scena, maschile"


In [49]:
#ESTRAIAMO SOLO I SOGGETTI MITOLOGICI 

import re

# 1. filtro base --> togliamo i NaN
df_arco = df_arco[df_arco["subject"].notna()].copy()

#creiamo la nostra lista di termini myth_terms
terms = [
    re.escape(t)
    for t in myth_terms
    if isinstance(t, str)
    and t.strip() != ""
    and t[0].isupper()  
]

# regex unica case-sensitive e per evitare termini composti
pattern = r'(?<![A-Za-zÀ-ÿ])(' + '|'.join(terms) + r')(?![A-Za-zÀ-ÿ])'
regex = re.compile(pattern)


#TERMINI DA ACCETTARE SOLO SE COMPAIONO INSIEME A QUESTE PAROLE []
context_rules = {
    "Ascanio" : ["fuoco", "cervo"],"Callisto": ["Arcade", "Diana"],"Concordia": ["Allegoria", "Innocenza", "Giustizia"],
    "Cornacchia": ["metamorfosi"],"Egeo": ["Teseo"],"Elena": ["ratto"],"Ettore": ["morte", "Protesilao", "Achille", "Andromaca", "Paride"],
    "Europa": ["ratto", "Toro", "sibilla", "mito","allegoria","ancelle","leone"],
    "Grazie" : ["tre", "danzanti"],"Ida" : ["Linceo"],"Ippolito":["episodi", "caduta", "morte"],
    "Nilo":["personificazione","allegoria"],"Roma":["dea"],"Silvano":["Dio"],"Sonno":["allegoria"],
    "Vesta":["Giove"],"Vittoria":["allegoria"]
}


# TERMINI DA ESCLUDERE SE COMPAIONO CON DETERMINATE PAROLE
exclusion_rules = {
    "Abbondanza": ["Santa", "Sant","san","papa"],"Amore": ["Dio", "Madonna", "castello", "fontana","antica"],"Andromeda":["galassia"],"Apollo":["sala","tempio"],
    "Atlante":["Scienze","turisti"],"Carità": ["Santa", "Sant", "Maria"],
    "Carita": ["Santa", "Sant", "Maria"],"Diana" : ["Beata", "duca","scenografia","tempio","conte","san"],
    "Dioniso": ["stemma","stemmi", "san", "santi", "s."],"Dionisio":["Ritratto","san","barletta","beati","vescovo","Santi","stemma","papa","padre","beato"],
    "Elio" : ["ponte","ritratto","paesaggio", "veduta"],"Enea": ["Beato", "discepolo","stemma", "papa", "silvio"],
    "Ercole": ["Ricotti", "Dandini", "Comini", "Farnese","cardinale", "tempio", "stemma", "ritratto", "Roccotti", "colonne", "porto", "basilica", "santa"],
    "Ermes" : ["ritratto"],"Esculapio": ["tempio"],"Satiro":["san","santi","santo","fratello"],
    "Fede" : ["santi", "sant", "san", "dottrina", "biblici", "Francia", "cattolica", "madonna", "evangelisti", "religione", "dio", "angeli", "angioletti","santa", "papa", "santo", "cristiana", "chiesa", "miracolo", "misericordia","virtù", "ritratto", "stemma"],
    "Flora" :["ritratto", "sante", "circo", "ruota", "iside", "madonna", "tempio"],"Galatea":["ritratto"],
    "Giasone":["san","ritratto"],"Giove":["san","tempio"],"Musa":["tempio"],
    "Giunone":["scenografia", "tempio"],"Giustizia" : ["porta", "palazzo", "misericordia", "san", "santa", "papa", "imperatore", "papale","Virtù", "apostoli","papa","Giglio"],
    "Gorgone" : ["santa"],"greco":["testo","teatro"],"Io": ["ritratti "],"Ippolito":["Nievo"],"Lavinia":["ritratto", "autoritratto"],
    "Leandro" : ["autoritratto","ritratto","stemma","San", "busto", "torre"],"Marte":["tempio","sala","ultore"],
    "Medea":["Colleoni"],"Mercurio":["tempio","san","assiso"],"Mirra":["San"],"Minerva":["autoritratto","chiesa","tempio"],
    "Narciso":["san"],"Nereo":["ritratto","san","santi","SS."],"Nettuno":["veduta"],"Oceano":["Atlantico"],
    "Oreste":["Ritratto","autoritratto"],"Pan":["zucchero","veduta"],"Patroclo":["san"],"Pirro":["Stipiciano"],"Pluto":["Topolino","Disney"],
    "Polidoro":["ritratto","Vicolo"],"Polissena":["Ritratto","santa"],"Priamo":["san","santo","Sulis","Fumagalli"],"Scilla":["topografica","veduta","paesaggio"],
    "Telemaco":["stemma","ritratto"],"Teseo":["Spirito"],"Tolomeo":["San","Santi","Ritratto","beato"],"Urano":["Monte"],
    "Venere":["Giorgio","Castello","modella","tempio"],"Verità":["stemma","Pierantonio","ritratto"],"Vittorie":["reggitarga"],
    "Speranza":["ritratto","santa","chiesa","papa","san","santo","madonna"],"Scilla":["ritratto"],"Serapide":["tempio"],"Achille":["stemma","ritratto","duomo","san"],
    "Adone":["sant"],"Cibele":["tempio"],"Cirene":["simone","palazzo"],"Teseo":["tempio","ritratto"],"Titano":["san"],"Ulisse":["ritratto","Dini","cardinalizia"],"Zeus":["tempio"]

}


def validate_matches(row):
    subject = row["subject"]
    valid_matches = []

    for match in row["matched_term"]:

        # PRIMO CHECK: controlla se il termine è nelle exclusion_rules
        if match in exclusion_rules:
            exclusion_keywords = exclusion_rules[match]
            # se almeno una parola di esclusione è presente, scarta il termine
            if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in exclusion_keywords):
                continue  
        
        # SECONDO CHECK: controlla context_rules 
        # se il termine non ha regole → tienilo
        if match not in context_rules:
            valid_matches.append(match)
            continue

        # controlla parole contestuali se il match è ambiguo 
        keywords = context_rules[match]

        if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in keywords):
            valid_matches.append(match)

    return valid_matches


df_arco["matched_term"] = df_arco["subject"].str.findall(regex) #trova tutti i termini e restituisce la lista
df_arco["matched_term"] = df_arco.apply(validate_matches, axis=1) #fa i controlli definiti sopra 

# df che risulta
df_matches = df_arco[df_arco["matched_term"].str.len() > 0]

In [50]:
# Verifica i risultati
print(f"Totale beni Arco: {len(df_arco)}")
print(f"Beni con match mitologico: {len(df_matches)}")

Totale beni Arco: 522846
Beni con match mitologico: 9672


In [51]:
#ulteriore iterazione NON case sensitive 
extra_check_terms = [
    "Aracne", "Argo", "Argonauti", "Arpia", "Arpie","Ascanio", "Callisto", "Centauro", "Cerva di Cerinea", "ciclope", "ciclopi", 
    "Chimera", "Cinghiale calidonio", "Cinghiale di Erimanto","Cupido", "Didone", "Elena", "Ippocampo",
    "Ippolito", "idra", "Minotauro", "musa", "muse", "Ratto di Europa","Sibilla samia", "Sibilla eritrea", "Sibilla tiburtina","Sirena", "Siringa", "Tritone", "satiro", "satiri"
]


extra_terms = [re.escape(t) for t in extra_check_terms]  #mette in sicurezza le parole per la regex 

extra_pattern = r'(?<![A-Za-zÀ-ÿ])(' + '|'.join(extra_terms) + r')(?![A-Za-zÀ-ÿ])'  
extra_regex = re.compile(extra_pattern, re.IGNORECASE)  #non case sensitive

#creo una copia del dataframe
df_matches = df_matches.copy()

# converte lista in set per confronto veloce
df_matches["matched_term_set"] = df_matches["matched_term"].apply(
    lambda x: set(x) if isinstance(x, list) else set()
)

#nuovo check
def find_new_matches(row):
    subject = row["subject"]
    already_found = row["matched_term_set"]

    all_hits = set(extra_regex.findall(subject))

    # rimuove quelli già trovati nel primo passaggio
    new_hits = all_hits - already_found

    return list(new_hits)


#applichiamo la funzione e creiamo una nuova colonna con gli extra_matched_terms
df_matches["extra_matched_terms"] = df_matches.apply(find_new_matches, axis=1)

#teniamo solo le righe utili
df_extra_hits = df_matches[df_matches["extra_matched_terms"].str.len() > 0].copy()


In [52]:
#normalizziamo i termini per poterli confrontare 
def normalize(s):
    return s.lower().strip() if isinstance(s, str) else ""

def merge_terms(row):
    base = row["matched_term"] if isinstance(row["matched_term"], list) else []
    extra = row["extra_matched_terms"] if isinstance(row["extra_matched_terms"], list) else []

    base_norm = {normalize(x): x for x in base}
    extra_norm = {normalize(x): x for x in extra}

    final = set()

    # 1. tengo gli extra che sono più specifici (es. europa --> ratto d'europa)
    for k, v in extra_norm.items():
        final.add(v)

    for b_key, b_val in base_norm.items():
        is_redundant = False

        for e_key in extra_norm:
            if b_key in e_key and b_key != e_key:
                is_redundant = True
                break

        if not is_redundant:
            final.add(b_val)  #tengo i termini base solo se non c'è una versione più specifica

    return list(final)

#applichiamo la funzione per avere i matched_term completi e più specifici
df_matches["final_terms"] = df_matches.apply(merge_terms, axis=1)

#df_matches[["subject", "final_terms"]].head()
df_checked = df_matches.sort_values(by="final_terms")[["item","label","subject", "final_terms"]]


In [53]:
print(len(df_checked))

9672


In [54]:
#esportiamo i risultati in un file csv

df_checked.to_csv("arco_myth_matches.csv", index=False, encoding="utf-8")


In [55]:
arco_endpoint = "https://dati.cultura.gov.it/sparql"
sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

df_base = pd.read_csv("arco_myth_matches.csv", encoding="utf-8")
items = df_base["item"].dropna().unique().tolist() #estraggo la lista di item unici per poi fare ulteriori query su Wikidata

In [56]:
# funzione per eseguire query in batch su SPARQL

def run_batched_query(items_list, query_template, batch_size=100):

    all_dfs = []

    for i in range(0, len(items_list), batch_size):

        batch = items_list[i:i + batch_size]
        formatted_items = " ".join(f"<{uri}>" for uri in batch) 

        query = query_template.replace("{ITEMS}", formatted_items)

        sparql_arco.setQuery(query)

        try:
            results = sparql_arco.query().convert()

            rows = []
            for r in results["results"]["bindings"]:
                rows.append({k: v["value"] for k, v in r.items()})

            if rows:
                all_dfs.append(pd.DataFrame(rows))

        except Exception as e:
            print(f"Errore batch {i}: {e}")

    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)

    return pd.DataFrame()

In [57]:
#QUERY 1: identifier, creators, types

query_ict= """ 
        PREFIX arco: <https://w3id.org/arco/ontology/arco/>
        PREFIX a-dd: <https://w3id.org/arco/ontology/denotative-description/>
        PREFIX a-loc: <https://w3id.org/arco/ontology/location/>
        PREFIX dc: <http://purl.org/dc/elements/1.1/>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        
        SELECT ?item ?identifier      
            (GROUP_CONCAT(DISTINCT ?creator; separator=", ") AS ?creators) 
            (SAMPLE (?creatorL) AS ?creatorLabel) 

            (GROUP_CONCAT(DISTINCT ?type; separator=", ") AS ?types) 
            (SAMPLE (?typeL) AS ?typeLabel) 

            (GROUP_CONCAT(DISTINCT ?instituteOrSite; separator=", ") AS ?institutesOrSites)
            (SAMPLE (?instituteOrSiteL) AS ?institutesOrSiteLabel) 
            
            (GROUP_CONCAT(DISTINCT ?place; separator=", ") AS ?coverage)
            (SAMPLE (?placeL) AS ?coverageLabel)
        
        WHERE {
            VALUES ?item {{ITEMS}} .
            ?item arco:uniqueIdentifier ?identifier .
 
            OPTIONAL {
                ?item dc:creator ?creator . 
                OPTIONAL {?creator rdfs:label ?creatorL .}
                }

            OPTIONAL {
                ?item a-dd:hasCulturalPropertyType ?type . 
                OPTIONAL {
                    ?type rdfs:label ?typeL .
                    FILTER(LANG(?typeL) = "it")
                    FILTER (!CONTAINS(STR(?typeL), "Tipo del bene:"))       
                }
            } 

            OPTIONAL {
                ?item a-loc:hasCulturalInstituteOrSite ?instituteOrSite . 
                OPTIONAL {?instituteOrSite rdfs:label ?instituteOrSiteL .}
                }

            OPTIONAL {
                ?item dc:coverage ?place .
                OPTIONAL {?place rdfs:label ?placeL .}
            } 
        }    
        
 
        GROUP BY ?item ?identifier
        """

In [58]:
#Q2 creation - date range 

query_date = """ 
        PREFIX arco: <https://w3id.org/arco/ontology/arco/>
        PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

        SELECT ?item 
            (SAMPLE(STR(?event)) AS ?event)
            (SAMPLE(STR(?startDate)) AS ?start) 
            (SAMPLE(STR(?endDate)) AS ?end)
        WHERE {
            VALUES ?item 
                {{ITEMS}}
    
        OPTIONAL {
            ?item a-cd:hasDating ?dating .
            ?dating a-cd:hasDatingEvent ?event .
            ?event a-cd:specificTime ?time .
            ?time arco:startTime ?startDate ;
              arco:endTime ?endDate .
        
            # Filtro per assicurarci di prendere solo la creazione
            FILTER(REGEX(STR(?event), "creation", "i"))
        }        
    }
    GROUP BY ?item 
        """

In [59]:
#QUERY 3 - geometries and locations 
query_location = """ 
        PREFIX a-loc: <https://w3id.org/arco/ontology/location/>
        PREFIX clvapit: <https://w3id.org/italia/onto/CLV/>
        SELECT ?item
            (SAMPLE(STR(?geometry)) AS ?geometry) 
            (SAMPLE(STR(?lat)) AS ?lat)
            (SAMPLE(STR(?long)) AS ?long)
        WHERE {
            VALUES ?item {{ITEMS}} .
            # OPTIONAL per coordinate geografiche
            OPTIONAL {
                ?item clvapit:hasGeometry ?geometry .
                ?geometry a-loc:hasCoordinates ?coordinates .
                ?coordinates a-loc:lat ?lat ;
                            a-loc:long ?long .
            }        
        }
        GROUP BY ?item 
        """

In [60]:
#QUERY 4 - images 
query_img = """ 
        PREFIX foaf: <http://xmlns.com/foaf/0.1/>
 
        SELECT ?item (SAMPLE(?image) AS ?sampleImage) #  solo una tra quelle disponibili
        WHERE {
            VALUES ?item {{ITEMS}} .
            OPTIONAL {?item foaf:depiction ?image . }        
        }
	GROUP BY ?item
        """

In [61]:
#eseguo le query 
df_ict = run_batched_query(items, query_ict, batch_size=50)


In [62]:
df_date = run_batched_query(items, query_date,batch_size=50)

In [63]:
df_location = run_batched_query(items, query_location, batch_size=50)

Errore batch 450: <urlopen error [WinError 10060] Impossibile stabilire la connessione. Risposta non corretta della parte connessa dopo l'intervallo di tempo oppure mancata risposta dall'host collegato>


In [64]:
df_img = run_batched_query(items, query_img, batch_size=50)

In [65]:
#creo un df unico 
df_enriched = df_base \
    .merge(df_ict, on="item", how="left") \
    .merge(df_date, on="item", how="left") \
    .merge(df_location, on="item", how="left") \
    .merge(df_img, on="item", how="left")


In [66]:
df_enriched.to_csv("arco_myth_enriched.csv", index=False)
print(len(df_enriched))

9672


In [67]:
#VISUALIZZIAMO IL DF ENRICHED 

df =pd.read_csv("arco_myth_enriched.csv")
df.head(1)

,item,label,subject,final_terms,identifier,creators,types,typeLabel,institutesOrSites,institutesOrSiteLabel,coverage,creatorLabel,event,start,end,geometry,lat,long,sampleImage
0,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza (decorazione plastica) di Brandani ...,Abbondanza,['Abbondanza'],1100209342,https://w3id.org/arco/resource/Agent/35421655f...,https://w3id.org/arco/resource/CulturalPropert...,decorazione plastica,NaN,NaN,Senigallia (AN),Brandani Federico (e Aiuti),https://w3id.org/arco/resource/Event/110020934...,ca 1560,ca 1562,https://w3id.org/arco/resource/Geometry/110020...,43.714963,13.219583,https://sigecweb.beniculturali.it/images/fulls...


In [68]:
#VISUALIZZIAMO SOLO LE LABEL DEL DF 

df = pd.read_csv("arco_myth_enriched.csv")

colonne_selezionate = [
    "item",
    "label",
    "subject",
    "final_terms",
    "typeLabel",
    "creatorLabel",
    "institutesOrSiteLabel",
    "coverage",
    "start",
    "end"
]

df_label = df[colonne_selezionate].copy()
df_label.head(5)

,item,label,subject,final_terms,typeLabel,creatorLabel,institutesOrSiteLabel,coverage,start,end
0,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza (decorazione plastica) di Brandani ...,Abbondanza,['Abbondanza'],decorazione plastica,Brandani Federico (e Aiuti),NaN,Senigallia (AN),ca 1560,ca 1562
1,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza (decorazione a intaglio) - bottega ...,Abbondanza,['Abbondanza'],decorazione a intaglio,NaN,Museo degli Argenti,Firenze (FI),(?) 1700,(?) 1786
2,https://w3id.org/arco/resource/HistoricOrArtis...,"allegoria dell'Abbondanza (candeliere, pendant...",allegoria dell'Abbondanza,['Abbondanza'],candeliere,NaN,NaN,Torino (TO),post 1840,ante 1860
3,https://w3id.org/arco/resource/Lombardia/Histo...,Allegoria dell'Abbondanza (disegno) - ambito l...,Allegoria dell'Abbondanza,['Abbondanza'],NaN,NaN,NaN,NaN,post 1700,ante 1710
4,https://w3id.org/arco/resource/HistoricOrArtis...,allegoria della Forza e dell'Abbondanza (scult...,allegoria della Forza e dell'Abbondanza,['Abbondanza'],scultura,Piazzetta Giacomo - 1640 ca./ 1705,NaN,Adria (RO),1683,1887


In [69]:
# ADESSO PROVIAMO A FARE ENTITY ENRICHMENT CON WIKIDATA ! ! ! ! 

In [70]:
#dividiamo i final_terms
df_terms = df_matches.explode("final_terms")


In [71]:
#creiamo un vocabolario con le entità trovate
unique_terms = (
    df_terms["final_terms"]
    .dropna()
    .str.lower()
    .unique()
    
)

terms_set = set(unique_terms)
print(f"soggetti unici normalizzati: {len(terms_set)}")

soggetti unici normalizzati: 468


In [110]:
print(terms_set)

{'euritione', 'iole', 'astrea', 'anassarete', 'ratto di europa', 'perseo', 'cloto', 'cimodocea', 'ascalabo', 'almone', 'centauro', 'bitone', 'eco', 'arione', 'efialte', 'ciparisso', 'galeso', 'ippocampo', 'pegaso', 'zete', 'anfione', 'achille', 'vespero', 'icaro', 'fetonte', 'protesilao', 'armonia', 'priapo', 'campi elisi', 'caco', 'europa', 'calipso', 'dioniso', 'perifante', 'rea', 'orfeo', 'paride', 'pentesilea', 'sibilla eritrea', 'didone', 'dionisio', 'edipo', 'pilade', 'deianira', 'atropo', 'bacco', 'calliope', 'elettra', 'tirreno', 'biblide', 'oto', 'enomao', 'vittorie', 'mida', 'eubuleo', 'enea', 'clizia', 'fortuna', 'cerva di cerinea', 'olimpo', 'lupa capitolina', 'dea roma', 'fedra', 'iapige', 'helios', 'eridano', 'ares', 'fineo', 'tritone', 'giustizia', 'artemide', 'verità', 'ammone', 'nefele', 'arianna', 'morfeo', 'igea', 'zeffiro', 'satiri', 'cadmo', 'endimione', 'alcesti', 'le fatiche di ercole', 'calais', 'atlante', 'tesifone', 'linceo', 'athena', 'anchise', 'plutone', 'a

In [72]:
#dobbiamo recuperare gli uri dal df di wikidata 
#prima dobbiamo splittare gli aliases 

def alias_match(alias_string, terms_set):

    if pd.isna(alias_string):
        return False

    aliases = [a.strip().lower() for a in alias_string.split(",")] #divido alias separati da ,

    return any(alias in terms_set for alias in aliases)

In [73]:
df_wikidata_filtered = df_wikidata[
    (
        # match diretto sulla label
        df_wikidata["label"]
        .str.strip()
        .str.lower()
        .isin(terms_set)
    )
    |
    (
        # match sugli aliases
        df_wikidata["aliases"].apply(
            lambda x: alias_match(x, terms_set)
        )
    )
].copy()


print(f"soggetti wikidata che matchano i termini unici: {len(df_wikidata_filtered)}")

df_wikidata_filtered[["entity", "label", "aliases"]].head()

#sono meno perchè aliases e label sono la stessa entity, quindi Nike e Vittoria Alata = 1 soggetto wikidata 
#sono meno perchè 'ciclope','ciclopi','idra','musa','muse','putti','putto','satiri','satiro' li ho messi io a mano nel secondo check sui soggetti. Non ci sono nel csv di wikidata 

soggetti wikidata che matchano i termini unici: 431


,entity,label,aliases
5,http://www.wikidata.org/entity/Q391379,Acheloo,NaN
14,http://www.wikidata.org/entity/Q163920,Adone,Atunis
19,http://www.wikidata.org/entity/Q35500,Afrodite,"Citerèa, Citerea, Aφροδίτη, Cipride"
26,http://www.wikidata.org/entity/Q4439972,Aglaia,NaN
35,http://www.wikidata.org/entity/Q911084,Alfeo,NaN


In [ ]:
#CONTROLLO PER VEDERE SE TUTTO TORNA 

wikidata_labels = set(
    df_wikidata_filtered["label"]
    .dropna()
    .str.strip()
    .str.lower()
)


wikidata_aliases = set()

for aliases in df_wikidata_filtered["aliases"].dropna():

    split_aliases = [
        a.strip().lower()
        for a in aliases.split(",")
    ]

    wikidata_aliases.update(split_aliases)

wikidata_terms = wikidata_labels.union(wikidata_aliases)

matched_terms = terms_set.intersection(wikidata_terms)
missing_terms = terms_set - wikidata_terms

print("TERMINI ARCO TOTALI:", len(terms_set))
print("TERMINI MATCHATI:", len(matched_terms))
print("TERMINI MANCANTI:", len(missing_terms))

missing_terms_sorted = sorted(missing_terms)

TERMINI ARCO TOTALI: 468
TERMINI MATCHATI: 468
TERMINI MANCANTI: 0


[]

In [ ]:
#proprietà che vogliamo estrarre (piccolo test per ora)
#instance of (P31)
#sex or gender (P21)
#domain of saint or deity (P2925)
#said to be the same as (P460)


In [94]:
# PROVIAMO A ESTRARRE LE MALEDETTE PROPRIETA CON QUESTO METODO PER NON FARCI BANNARE DA WIKIDATA

#iniziamo estraendo i QID delle entità che ci interessano 
df_wikidata_filtered = df_wikidata_filtered.copy()
df_wikidata_filtered["qid"] = (
    df_wikidata_filtered["entity"]
    .str.split("/")
    .str[-1]
)

#mettiamoli i una lista 

qids = df_wikidata_filtered["qid"].dropna().unique().tolist()

print(len(qids))

431


In [95]:
headers = {
    "User-Agent": "ArCo-MythProject/1.0 (contact: anna.pasetto2@studio.unibo.it)"
}

In [96]:
label_cache = {}

def get_label(qid):

    if qid is None:
        return None

    if qid in label_cache:
        return label_cache[qid]

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

    try:
        r = requests.get(url, headers=headers, timeout=10)
        data = r.json()

        label = (
            data["entities"][qid]["labels"]
            .get("en", {})
            .get("value")
        )

    except:
        label = None

    label_cache[qid] = label
    return label

In [97]:
#selezioniamo solo gli instance_of che ci interessano per poterli estrarre

preferred_p31 = {
    "Q11688446",
    "Q22988604",
    "Q111252729",
    "Q13405593",
    "Q24336068",
    "Q22989102",
    "Q20902363",
    "Q124940323",
    "Q24334685"
}

In [98]:
#visto che non possiamo fare SPARQL Queries, prendiamo direttamente tutta la scheda wikidata di ogni entità (ho stra paura)

def get_wikidata_properties(qid):

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

    try:
        r = requests.get(url, headers=headers, timeout=(5, 15))
        r.raise_for_status()
        data = r.json()

    except Exception as e:
        print("Errore su", qid, e)
        return {
            "qid": qid,
            "P31": None,
            "P21": None,
            "P2925": None,
            "P460": None,
            "P1889": None,
        }

    entity = data["entities"][qid]
    claims = entity.get("claims", {})

    #funzione generica per prendere il primo valore 
    def extract(prop):

        if prop not in claims:
            return None

        try:
            value = claims[prop][0]["mainsnak"]["datavalue"]["value"]

            if isinstance(value, dict) and "id" in value:
                return value["id"]

            return value

        except:
            return None

    #funzione separata per p31

    def extract_p31():

        if "P31" not in claims:
            return None

        found = []

        for claim in claims["P31"]:

            try:

                value = claim["mainsnak"]["datavalue"]["value"]

                if isinstance(value, dict) and "id" in value:

                    qid_value = value["id"]

                    found.append(qid_value)

                    # se troviamo uno dei tipi preferiti
                    if qid_value in preferred_p31:
                        return qid_value

            except:
                continue

        if found:
            return found[0]

        return None

    return {
        "qid": qid,
        "P31": extract("P31"),
        "P21": extract("P21"),
        "P2925": extract("P2925"),
        "P460": extract("P460"),
        "P1889": extract("P1889"), # different from
    }


In [99]:
#ora facciamo un LOOOOOP su tutte le entità 
results = []

for i, qid in enumerate(tqdm(qids)):

    results.append(get_wikidata_properties(qid))

    time.sleep(1)  

    # salva ogni 50 per sicurezza
    if i % 50 == 0 and i != 0:
        pd.DataFrame(results).to_csv("backup_wikidata.csv", index=False)
        print(f"Salvato checkpoint: {i}")


 12%|█▏        | 51/431 [01:26<10:16,  1.62s/it]

Salvato checkpoint: 50


 23%|██▎       | 101/431 [02:49<09:10,  1.67s/it]

Salvato checkpoint: 100


 35%|███▌      | 151/431 [04:19<09:40,  2.07s/it]

...

 93%|█████████▎| 401/431 [15:26<01:26,  2.89s/it]

Salvato checkpoint: 400


100%|█████████▉| 430/431 [16:45<00:02,  2.22s/it]

Errore su Satiro 400 Client Error: Bad Request for url: https://www.wikidata.org/wiki/Special:EntityData/Satiro.json


100%|██████████| 431/431 [16:47<00:00,  2.34s/it]


In [100]:
df_props = pd.DataFrame(results)
df_props.head(10)

,qid,P31,P21,P2925,P460,P1889
0,Q391379,Q3027575,Q6581097,Q203923,None,None
1,Q163920,Q22989102,Q6581097,Q7242,None,None
2,Q35500,Q22989102,Q6581072,Q316,Q47652,Q9141765
3,Q4439972,Q22989102,Q6581072,None,None,None
4,Q911084,Q3027575,Q6581097,Q941745,None,None
5,Q107785,Q182406,Q6581072,None,None,Q108904
6,Q3575327,Q22989102,Q6581097,None,None,None
7,Q180222,Q22989102,Q6581072,Q165,Q2987780,Q186663
8,Q572133,Q22989102,Q6581097,Q316,None,None
9,Q37340,Q22989102,Q6581097,None,Q900649,None


In [101]:
cols = ["P31","P21","P2925","P460", "P1889"] 

def decorate(qid):

    if qid is None:
        return None

    label = get_label(qid)

    if label:
        return f"{qid} ({label})"
    return qid

for col in cols:
    df_props[col] = df_props[col].apply(decorate)



In [102]:
df_props.head()

,qid,P31,P21,P2925,P460,P1889
0,Q391379,Q3027575 (Potamoi),Q6581097 (male),Q203923 (Achelous River),None,None
1,Q163920,Q22989102 (Greek deity),Q6581097 (male),Q7242 (beauty),None,None
2,Q35500,Q22989102 (Greek deity),Q6581072 (female),Q316 (love),Q47652 (Venus),Q9141765 (Afrodyta)
3,Q4439972,Q22989102 (Greek deity),Q6581072 (female),None,None,None
4,Q911084,Q3027575 (Potamoi),Q6581097 (male),Q941745 (Alfeios),None,None


In [103]:
#carichiamo tutto in un df 

df_props.to_csv("entities_and_properties.csv", index=False)

In [104]:
# estrazione delle proprietà con molti oggetti:
# wdt:P451 # unmarried partner
# wdt:P40 # child
# wdt:P26 #spouse
# wdt:P3373 #sibling
#father (P22)
#mother (P25)
#killed by (P157)

# def ...


In [105]:
valid_qids = set(df_wikidata_filtered["qid"])

network_properties = {
    "P22": "father",
    "P25": "mother",
    "P40": "child",
    "P3373": "sibling",
    "P26": "spouse",
    "P451" : "unmarried partner",
    "P157" : "killed_by"
}

In [106]:
#ORA PROVIAMO AD ESTRARRE I NETWORK 
def extract_relations(qid):

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

    try:
        r = requests.get(url, headers=headers, timeout=(5,15))
        r.raise_for_status()
        data = r.json()

    except Exception as e:
        print("Errore:", qid, e)
        return []

    entity = data["entities"][qid]
    claims = entity.get("claims", {})

    rows = []

    for prop, relation_name in network_properties.items():

        if prop not in claims:
            continue

        for claim in claims[prop]:

            try:
                value = claim["mainsnak"]["datavalue"]["value"]

                if isinstance(value, dict) and "id" in value:

                    target_qid = value["id"]

                    # tieni SOLO relazioni interne al corpus
                    if target_qid in valid_qids:

                        rows.append({
                            "source": qid,
                            "relation": relation_name,
                            "target": target_qid,
                            "target_label": get_label(target_qid)
                        })

            except:
                continue

    return rows

In [107]:
all_relations = []

for qid in tqdm(qids):

    rels = extract_relations(qid)

    all_relations.extend(rels)

    time.sleep(1)

df_relations = pd.DataFrame(all_relations)

100%|█████████▉| 430/431 [26:06<00:02,  2.09s/it]  

Errore: Satiro 400 Client Error: Bad Request for url: https://www.wikidata.org/wiki/Special:EntityData/Satiro.json


100%|██████████| 431/431 [26:08<00:00,  3.64s/it]


In [108]:
qid_to_label = dict(
    zip(
        df_wikidata_filtered["qid"],
        df_wikidata_filtered["label"]
    )
)

In [109]:
df_relations.to_csv("mythological_network.csv", index=False)